# 🗺️ GPS Route + Foot Pressure Visualisation
**Walk session – Kristian, April 24 2026**

This notebook:
1. Parses the Garmin `.gpx` file (GPS route, 1 Hz) 
2. Loads the Stappone insole IMU/pressure `.csv` file (~62 Hz, both feet)
3. Time-synchronises both streams (both use Unix ms timestamps)
4. Renders **two dynamic interactive maps**:
   - **Map 1** – plain GPS route coloured by elapsed time
   - **Map 2** – GPS route where each point is coloured by total foot pressure
5. Bonus time-series chart of left vs right foot pressure

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    USE_PLOTLY = True
    print('✅ Plotly available – full interactive maps enabled')
except ImportError:
    USE_PLOTLY = False
    import matplotlib.pyplot as plt
    import matplotlib.cm as mcm
    import matplotlib.colors as mc
    print('⚠️  Plotly not found – using matplotlib (static). Run: pip install plotly')

print('Libraries ready.')

✅ Plotly available – full interactive maps enabled
Libraries ready.


## 1 · Load & Parse GPX

In [ ]:
GPX_FILE = 'activity_22645980458.gpx'   
CSV_FILE = 'Kristian_dry_0425.csv'      

tree = ET.parse(GPX_FILE)
root = tree.getroot()
ns   = {'gpx': 'http://www.topografix.com/GPX/1/1'}

rows = []
for pt in root.findall('.//gpx:trkpt', ns):
    t_node = pt.find('gpx:time', ns)
    e_node = pt.find('gpx:ele',  ns)
    t = pd.to_datetime(t_node.text.strip(), utc=True)
    rows.append({
        'lat':      float(pt.attrib['lat']),
        'lon':      float(pt.attrib['lon']),
        'ele':      float(e_node.text) if e_node is not None else np.nan,
        'time_utc': t,
        # Convert to Unix milliseconds to match Stappone timestamps
        'ts_ms':    int(t.timestamp() * 1000),
    })

gdf = pd.DataFrame(rows)

print(f'GPX points  : {len(gdf)}')
print(f'Duration    : {gdf["time_utc"].iloc[-1] - gdf["time_utc"].iloc[0]}')
print(f'Bbox        : lat [{gdf.lat.min():.5f}, {gdf.lat.max():.5f}]  '
      f'lon [{gdf.lon.min():.5f}, {gdf.lon.max():.5f}]')
gdf.head(3)

GPX points  : 440
Duration    : 0 days 00:07:19
Bbox        : lat [42.33698, 42.34058]  lon [-71.09356, -71.09060]


,lat,lon,ele,time_utc,ts_ms
0,42.336990,-71.090600,3.0,2026-04-24 19:09:08+00:00,1777057748000
1,42.336982,-71.090596,3.0,2026-04-24 19:09:09+00:00,1777057749000
2,42.336983,-71.090596,3.0,2026-04-24 19:09:10+00:00,1777057750000


## 2 · Load & Process IMU / Pressure CSV

In [3]:
df = pd.read_csv(CSV_FILE)

# Stappone: sole_id 1 = right foot, 2 = left foot
# timestamp = Unix milliseconds since 01.01.1970 UTC
pressure_cols = [f'pressure_{i:02d}' for i in range(1, 13)]
df['total_pressure'] = df[pressure_cols].sum(axis=1)
df['time_utc']       = pd.to_datetime(df['timestamp'], unit='ms', utc=True)

right = df[df.sole_id == 1][['timestamp', 'time_utc', 'total_pressure'] + pressure_cols].copy()
left  = df[df.sole_id == 2][['timestamp', 'total_pressure'] + pressure_cols].copy()
right.columns = ['timestamp', 'time_utc', 'total_pressure_R'] + [f'{c}_R' for c in pressure_cols]
left.columns  = ['timestamp', 'total_pressure_L']              + [f'{c}_L' for c in pressure_cols]

imu = (pd.merge(right, left, on='timestamp', how='outer')
         .sort_values('timestamp').reset_index(drop=True))
imu['total_pressure'] = imu[['total_pressure_R', 'total_pressure_L']].mean(axis=1)
imu['time_utc']       = pd.to_datetime(imu['timestamp'], unit='ms', utc=True)
imu['ts_ms']          = imu['timestamp']  # already Unix ms

sr = 1000 / imu['ts_ms'].diff().median()
print(f'IMU rows      : {len(imu)} unique timestamps')
print(f'Sample rate   : ~{sr:.0f} Hz (both feet merged)')
print(f'Pressure range: {imu.total_pressure.min():.0f} – {imu.total_pressure.max():.0f} (raw ADC sum)')
imu[['time_utc', 'total_pressure_R', 'total_pressure_L', 'total_pressure']].head(4)

IMU rows      : 25634 unique timestamps
Sample rate   : ~62 Hz (both feet merged)
Pressure range: 3247 – 4377 (raw ADC sum)


,time_utc,total_pressure_R,total_pressure_L,total_pressure
0,2026-04-24 19:09:06.944000+00:00,3910.0,4108,4009.0
1,2026-04-24 19:09:06.960000+00:00,3913.0,4105,4009.0
2,2026-04-24 19:09:06.976000+00:00,3877.0,4105,3991.0
3,2026-04-24 19:09:06.992000+00:00,3871.0,4097,3984.0


## 3 · Time-Synchronise GPS ↔ Pressure

Both files use Unix millisecond timestamps.  
`merge_asof` assigns the nearest IMU reading (within 2 s) to each 1 Hz GPS point.

In [4]:
merged = pd.merge_asof(
    gdf.sort_values('ts_ms'),
    imu.sort_values('ts_ms')[['ts_ms', 'total_pressure', 'total_pressure_R', 'total_pressure_L']],
    on='ts_ms', direction='nearest', tolerance=2000
).reset_index(drop=True)

n_before = len(merged)
merged = merged.dropna(subset=['total_pressure']).reset_index(drop=True)
print(f'Matched GPS pts : {len(merged)} / {n_before}  '
      f'({n_before - len(merged)} outside IMU window)')

p_min, p_max = merged['total_pressure'].quantile(0.02), merged['total_pressure'].quantile(0.98)
merged['p_norm']    = ((merged['total_pressure'] - p_min) / (p_max - p_min)).clip(0, 1)
merged['elapsed_s'] = (merged['ts_ms'] - merged['ts_ms'].iloc[0]) / 1000

print(f'Pressure range  : {merged.total_pressure.min():.0f} – {merged.total_pressure.max():.0f}')
merged[['time_utc', 'lat', 'lon', 'total_pressure', 'total_pressure_R',
        'total_pressure_L', 'elapsed_s']].describe().round(1)

Matched GPS pts : 412 / 440  (28 outside IMU window)
Pressure range  : 3395 – 4346


,lat,lon,total_pressure,total_pressure_R,total_pressure_L,elapsed_s
count,412.0,412.0,412.0,412.0,412.0,412.0
mean,42.3,-71.1,3678.0,3564.4,3791.6,205.5
std,0.0,0.0,145.8,646.7,604.4,119.1
min,42.3,-71.1,3395.0,2646.0,2969.0,0.0
25%,42.3,-71.1,3592.4,2853.8,3144.0,102.8
50%,42.3,-71.1,3657.8,3768.5,3918.5,205.5
75%,42.3,-71.1,3743.8,4181.0,4368.2,308.2
max,42.3,-71.1,4346.5,4511.0,4681.0,411.0


## 4 · Map 1 – GPS Route (colour = elapsed time)

In [9]:
center_lat = merged['lat'].mean()
center_lon = merged['lon'].mean()

if USE_PLOTLY:
    fig1 = go.Figure()

    fig1.add_trace(go.Scattermapbox(
        lat=merged['lat'], lon=merged['lon'],
        mode='lines',
        line=dict(width=3, color='rgba(90,120,200,0.45)'),
        name='Route', hoverinfo='skip'
    ))

    fig1.add_trace(go.Scattermapbox(
        lat=merged['lat'], lon=merged['lon'],
        mode='markers',
        marker=dict(
            size=8, color=merged['elapsed_s'],
            colorscale='Viridis',
            colorbar=dict(title='Elapsed (s)', x=1.01),
            showscale=True,
        ),
        customdata=np.column_stack([
            merged['elapsed_s'],
            merged['time_utc'].dt.strftime('%H:%M:%S'),
            merged['ele'].fillna(0),
        ]),
        hovertemplate=(
            '<b>🕐 %{customdata[1]}</b>  (t+%{customdata[0]:.0f}s)<br>'
            '📍 (%{lat:.5f}, %{lon:.5f})<br>'
            '⛰ Elevation: %{customdata[2]:.1f} m'
            '<extra></extra>'
        ),
        name='GPS points'
    ))

    for label, idx, color in [('🟢 Start', 0, 'limegreen'), ('🔴 End', -1, 'crimson')]:
        fig1.add_trace(go.Scattermapbox(
            lat=[merged['lat'].iloc[idx]], lon=[merged['lon'].iloc[idx]],
            mode='markers+text', marker=dict(size=18, color=color),
            text=[label], textposition='top right', name=label
        ))

    fig1.update_layout(
        title=dict(text='Map 1 – GPS Route  (colour = elapsed time)', font=dict(size=16)),
        mapbox=dict(
    style='white-bg',
    layers=[{
        'below': 'traces',
        'sourcetype': 'raster',
        'source': ['https://tile.opentopomap.org/{z}/{x}/{y}.png'],
    }],
    center=dict(lat=center_lat, lon=center_lon),
    zoom=14,
),
        height=640, margin=dict(l=0, r=0, t=45, b=0),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)')
    )
    fig1.show()

else:
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    sc = ax1.scatter(merged['lon'], merged['lat'],
                     c=merged['elapsed_s'], cmap='viridis', s=20, zorder=3)
    ax1.plot(merged['lon'], merged['lat'], color='royalblue', lw=1, alpha=0.4, zorder=2)
    plt.colorbar(sc, ax=ax1, label='Elapsed time (s)')
    ax1.scatter(merged['lon'].iloc[0],  merged['lat'].iloc[0],  s=200, c='limegreen', zorder=5, label='Start')
    ax1.scatter(merged['lon'].iloc[-1], merged['lat'].iloc[-1], s=200, c='crimson',   zorder=5, label='End')
    ax1.legend(); ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude')
    ax1.set_title('Map 1 – GPS Route (colour = elapsed time)\nInstall plotly for an interactive map')
    plt.tight_layout(); plt.show()

## 5 · Map 2 – Route Coloured by Total Foot Pressure

Each GPS point is coloured by the **sum of all 12 pressure sensors** (left + right foot averaged).  
🔴 = high pressure &nbsp;|&nbsp; 🔵 = low pressure  
Hover over any point to see per-foot breakdown.

In [13]:
if USE_PLOTLY:
    fig2 = go.Figure()

    # Thin base line for continuity
    fig2.add_trace(go.Scattermapbox(
        lat=merged['lat'], lon=merged['lon'],
        mode='lines', line=dict(width=2, color='rgba(120,120,120,0.3)'),
        name='Route', hoverinfo='skip'
    ))

    # Pressure-coloured scatter (creates the coloured-line visual)
    fig2.add_trace(go.Scattermapbox(
        lat=merged['lat'], lon=merged['lon'],
        mode='markers',
        marker=dict(
            size=11,
            color=merged['total_pressure'],
            colorscale='RdYlBu_r',
            cmin=p_min, cmax=p_max,
            colorbar=dict(
                title='Total pressure<br>(ADC sum, both feet)',
                x=1.01, tickformat='.0f'
            ),
            showscale=True,
        ),
        customdata=np.column_stack([
            merged['total_pressure'].fillna(0),
            merged['total_pressure_R'].fillna(0),
            merged['total_pressure_L'].fillna(0),
            merged['elapsed_s'],
            merged['time_utc'].dt.strftime('%H:%M:%S'),
        ]),
        hovertemplate=(
            '<b>🕐 %{customdata[4]}</b>  (t+%{customdata[3]:.0f}s)<br>'
            '⚡ Total pressure : <b>%{customdata[0]:.0f}</b><br>'
            '👟 Right foot     : %{customdata[1]:.0f}<br>'
            '👟 Left foot      : %{customdata[2]:.0f}<br>'
            '📍 (%{lat:.5f}, %{lon:.5f})'
            '<extra></extra>'
        ),
        name='Pressure'
    ))

    for label, idx, color in [('🟢 Start', 0, 'limegreen'), ('🔴 End', -1, 'crimson')]:
        fig2.add_trace(go.Scattermapbox(
            lat=[merged['lat'].iloc[idx]], lon=[merged['lon'].iloc[idx]],
            mode='markers+text', marker=dict(size=18, color=color),
            text=[label], textposition='top right', name=label
        ))

    fig2.update_layout(
        title=dict(
            text='Map 2 – Route Coloured by Total Foot Pressure  (🔴 high | 🔵 low)',
            font=dict(size=16)
        ),
        mapbox=dict(
    style='white-bg',
    layers=[{
        'below': 'traces',
        'sourcetype': 'raster',
        'source': ['https://tile.opentopomap.org/{z}/{x}/{y}.png'],
    }],
    center=dict(lat=center_lat, lon=center_lon),
    zoom=14,
),
        height=650, margin=dict(l=0, r=0, t=45, b=0),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)')
    )
    fig2.show()

else:
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    cmap = mcm.get_cmap('RdYlBu_r')
    norm = mc.Normalize(vmin=p_min, vmax=p_max)
    lats, lons, pres = merged['lat'].values, merged['lon'].values, merged['total_pressure'].values
    for i in range(len(lats) - 1):
        ax2.plot([lons[i], lons[i+1]], [lats[i], lats[i+1]],
                 color=cmap(norm((pres[i] + pres[i+1]) / 2)), lw=4)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax2, label='Total pressure (raw ADC sum)')
    ax2.scatter(lons[0],  lats[0],  s=200, c='limegreen', zorder=5, label='Start')
    ax2.scatter(lons[-1], lats[-1], s=200, c='crimson',   zorder=5, label='End')
    ax2.legend(); ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
    ax2.set_title('Map 2 – Route Coloured by Total Foot Pressure')
    plt.tight_layout(); plt.show()

## 6 · Bonus – Left vs Right Foot Pressure Time-Series

In [14]:
if USE_PLOTLY:
    fig3 = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=['Right Foot Total Pressure (sole_id=1)',
                        'Left Foot Total Pressure (sole_id=2)'],
        vertical_spacing=0.08
    )
    fig3.add_trace(go.Scatter(
        x=imu['time_utc'], y=imu['total_pressure_R'],
        mode='lines', name='Right foot', line=dict(color='steelblue', width=0.8),
        hovertemplate='%{x|%H:%M:%S.%L} | Right: %{y:.0f}<extra></extra>'
    ), row=1, col=1)
    fig3.add_trace(go.Scatter(
        x=imu['time_utc'], y=imu['total_pressure_L'],
        mode='lines', name='Left foot', line=dict(color='tomato', width=0.8),
        hovertemplate='%{x|%H:%M:%S.%L} | Left: %{y:.0f}<extra></extra>'
    ), row=2, col=1)
    fig3.update_yaxes(title_text='Pressure (raw ADC sum)', row=1, col=1)
    fig3.update_yaxes(title_text='Pressure (raw ADC sum)', row=2, col=1)
    fig3.update_xaxes(title_text='Time (UTC)', row=2, col=1)
    fig3.update_layout(
        title='Left vs Right Foot Pressure Over Time',
        height=520, template='plotly_white', hovermode='x unified'
    )
    fig3.show()

else:
    fig3, (ax3a, ax3b) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    ax3a.plot(imu['time_utc'], imu['total_pressure_R'], lw=0.6, color='steelblue')
    ax3b.plot(imu['time_utc'], imu['total_pressure_L'], lw=0.6, color='tomato')
    for ax, title in [(ax3a, 'Right foot (sole_id=1)'), (ax3b, 'Left foot (sole_id=2)')]:
        ax.set_ylabel('Pressure (raw ADC)'); ax.set_title(title)
    ax3b.set_xlabel('Time (UTC)')
    plt.tight_layout(); plt.show()

## 7 · Export Merged Dataset

In [15]:
out = merged[['time_utc', 'lat', 'lon', 'ele', 'elapsed_s',
              'total_pressure', 'total_pressure_R', 'total_pressure_L', 'p_norm']].copy()
out.to_csv('route_pressure_merged.csv', index=False)
print(f'✅ Saved → route_pressure_merged.csv  ({len(out)} rows)')
out.head()

✅ Saved → route_pressure_merged.csv  (412 rows)


,time_utc,lat,lon,ele,elapsed_s,total_pressure,total_pressure_R,total_pressure_L,p_norm
0,2026-04-24 19:09:08+00:00,42.336990,-71.090600,3.0,0.0,3935.5,3809.0,4062.0,0.677325
1,2026-04-24 19:09:09+00:00,42.336982,-71.090596,3.0,1.0,3675.5,3919.0,3432.0,0.307676
2,2026-04-24 19:09:10+00:00,42.336983,-71.090596,3.0,2.0,3592.5,4173.0,3012.0,0.189673
3,2026-04-24 19:09:11+00:00,42.336983,-71.090606,3.0,3.0,3531.0,2840.0,4222.0,0.102236
4,2026-04-24 19:09:12+00:00,42.336989,-71.090607,3.0,4.0,3415.5,2683.0,4148.0,0.000000
